# Project 3: Sales Data Filter & Report Engine
**NumPy | Google Colab | Intermediate**

## Objective
Simulate one year of sales for 10 stores and 12 months, then build a small query engine that filters records and generates monthly, store-wise, and quarterly reports.

## NumPy topics
`np.arange`, `np.linspace`, `np.random`, `reshape`, Boolean masking (`&`, `|`, `~`), fancy indexing, and 2D slicing.

**Note:** The data is synthetic (generated for learning). Results are calculated when the notebook runs; no benchmark or sales values are pre-invented.

## 1. Import libraries and set reproducible randomness

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

## 2. Create month labels and a 10 × 12 sales array
Each row represents a store and each column represents a month. Sales values are simulated whole-number units.

In [ ]:
months = np.arange(1, 13)
month_names = np.array([
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
])
stores = np.arange(1, 11)

# Simulated monthly sales for 10 stores, in sales units.
sales = np.random.randint(500, 5000, size=(10, 12))

print("Sales array shape:", sales.shape)
print("Dimensions:", sales.ndim)
print("Total values:", sales.size)
print("\nSales array (rows=stores, columns=months):")
print(sales)

## 3. Basic summaries

In [ ]:
monthly_totals = sales.sum(axis=0)
store_totals = sales.sum(axis=1)
monthly_average = sales.mean(axis=0)
store_average = sales.mean(axis=1)
grand_total = sales.sum()

print("Total simulated annual sales:", int(grand_total))
print("Average monthly sales across all stores:", round(float(sales.mean()), 2))
print("\nMonthly totals:")
for name, total in zip(month_names, monthly_totals):
    print(f"{name}: {int(total)}")

print("\nStore-wise annual totals:")
for store_id, total in zip(stores, store_totals):
    print(f"Store {store_id}: {int(total)}")

## 4. Slice quarters (Q1–Q4) and individual stores
Python's stop index is exclusive, so each quarter contains three columns.

In [ ]:
quarters = {
    "Q1 (Jan-Mar)": sales[:, 0:3],
    "Q2 (Apr-Jun)": sales[:, 3:6],
    "Q3 (Jul-Sep)": sales[:, 6:9],
    "Q4 (Oct-Dec)": sales[:, 9:12],
}

print("Quarter totals across all stores:")
for quarter_name, quarter_data in quarters.items():
    print(f"{quarter_name}: {int(quarter_data.sum())}")

store_3 = sales[2, :]  # third store, all 12 months
print("\nStore 3 monthly sales:", store_3)
print("First 3 stores and first 4 months:\n", sales[0:3, 0:4])

## 5. Boolean-mask queries
The target below is an adjustable example threshold, not a real-world business target. `&`, `|`, and `~` combine or invert Boolean conditions.

In [ ]:
target = 3000

# Individual store-month entries above the target
above_target_mask = sales > target
above_target_entries = sales[above_target_mask]
print(f"Entries above {target}:", int(above_target_entries.size))
print("Values above target:", above_target_entries)

# Months whose total sales across all stores exceed the monthly average
monthly_mask = monthly_totals > monthly_totals.mean()
print("\nMonths above average total sales:")
print(month_names[monthly_mask])

# Stores whose annual total is below the average annual total per store
below_average_store_mask = store_totals < store_totals.mean()
print("\nStores below average annual sales:")
print(stores[below_average_store_mask])

# Combined conditions: entries above target AND in months Jul-Dec
second_half_mask = np.arange(12) >= 6
combined_mask = (sales > target) & second_half_mask[np.newaxis, :]
print("\nAbove-target entries in Jul-Dec:", int(combined_mask.sum()))

# OR condition: low sales (<1000) or high sales (>4000)
extreme_mask = (sales < 1000) | (sales > 4000)
print("Low-or-high sales entries:", int(extreme_mask.sum()))

# NOT condition: entries not above target
not_above_target_mask = ~(sales > target)
print("Entries at or below target:", int(not_above_target_mask.sum()))

## 6. Fancy indexing
Select stores 2, 5, and 8 (zero-based indices `[1, 4, 7]`) and identify the three best months by total sales.

In [ ]:
selected_indices = np.array([1, 4, 7])
selected_stores = sales[selected_indices, :]
print("Selected stores (IDs 2, 5, 8):\n", selected_stores)

top3_month_indices = np.argsort(monthly_totals)[-3:][::-1]
print("\nTop 3 months:")
for idx in top3_month_indices:
    print(f"{month_names[idx]}: {int(monthly_totals[idx])}")

top3_month_sales = sales[:, top3_month_indices]
print("\nSales for all stores in the top 3 months (columns in top-month order):\n",
      top3_month_sales)

## 7. Growth targets using `np.linspace`
The growth targets are illustrative thresholds from 2,000 to 3,000 units. They are not forecasts; they simply demonstrate `np.linspace` and comparison masks.

In [ ]:
growth_targets = np.linspace(2000, 3000, num=12)
target_met = monthly_totals >= growth_targets

target_report = pd.DataFrame({
    "Month": month_names,
    "Monthly total": monthly_totals,
    "Illustrative target": np.round(growth_targets, 2),
    "Target met": target_met
})
target_report

## 8. Monthly and store-wise report tables

In [ ]:
monthly_report = pd.DataFrame({
    "Month": month_names,
    "Total sales": monthly_totals,
    "Average per store": np.round(monthly_average, 2)
})

store_report = pd.DataFrame({
    "Store": stores,
    "Annual total": store_totals,
    "Monthly average": np.round(store_average, 2),
    "Below store average": below_average_store_mask
})

print("MONTHLY REPORT")
display(monthly_report)
print("STORE-WISE REPORT")
display(store_report)

## 9. Visualize the reports

In [ ]:
plt.figure(figsize=(11, 4))
plt.bar(month_names, monthly_totals)
plt.title("Total Sales by Month (10 Simulated Stores)")
plt.xlabel("Month")
plt.ylabel("Sales units")
plt.grid(axis="y", alpha=0.25)
plt.show()

plt.figure(figsize=(11, 4))
plt.bar(stores.astype(str), store_totals)
plt.title("Annual Sales by Store")
plt.xlabel("Store")
plt.ylabel("Sales units")
plt.grid(axis="y", alpha=0.25)
plt.show()

## 10. Quick query function
Call `query_sales()` with a threshold and optionally a store ID to inspect matching entries.

In [ ]:
def query_sales(data, threshold=3000, store_id=None, comparison="greater"):
    \"\"\"Return sales entries matching a threshold, optionally for one store.

    store_id is 1-based (1 through 10).
    comparison may be 'greater' or 'less'.
    \"\"\"
    if data.shape != (10, 12):
        raise ValueError("Expected a sales array with shape (10, 12).")
    if store_id is not None and not 1 <= store_id <= 10:
        raise ValueError("store_id must be between 1 and 10.")
    if comparison not in {"greater", "less"}:
        raise ValueError("comparison must be 'greater' or 'less'.")

    rows = data if store_id is None else data[store_id - 1:store_id, :]
    mask = rows > threshold if comparison == "greater" else rows < threshold
    return rows[mask]

print("All entries above 4,000:", query_sales(sales, threshold=4000))
print("Store 1 entries below 1,000:", query_sales(sales, threshold=1000,
                                                  store_id=1, comparison="less"))

## 11. Validation checks
These checks verify array dimensions, report lengths, and query counts. They should pass whenever the notebook runs successfully.

In [ ]:
assert sales.shape == (10, 12)
assert monthly_totals.shape == (12,)
assert store_totals.shape == (10,)
assert sum(q.sum() for q in quarters.values()) == sales.sum()
assert int(above_target_mask.sum() + not_above_target_mask.sum()) == sales.size
assert len(monthly_report) == 12
assert len(store_report) == 10

print("All validation checks passed.")

## Conclusion
This project demonstrates how NumPy arrays can represent store-month sales data and how slicing, Boolean masks, and fancy indexing can answer simple reporting questions. Pandas is used only to present tabular reports, and Matplotlib visualizes the monthly and store-wise totals.

## Resume bullet (use only after you have run and reviewed the notebook)
Built a NumPy-based sales reporting engine for a 10-store, 12-month synthetic dataset using quarterly slicing, combined Boolean masks, fancy indexing, and monthly/store-wise reports.